## **📘DimDate — Calendar Dimension** ##
### **Purpose** ###

- Generate a reusable date dimension aligned to your Silver date range.

**Source**

- outage_start_date

- outage_end_date

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# --------------------------------------------
# Determine date range from Silver
# --------------------------------------------

silver_df = spark.table("lh_Silver_Telecom.dbo.silver_network_events")

date_bounds = silver_df.select(
    min("outage_start_date").alias("min_date"),
    max("outage_start_date").alias("max_date")
).collect()[0]

start_date = date_bounds["min_date"]
end_date = date_bounds["max_date"]

print(f"Generating dim_date from {start_date} to {end_date}")

# --------------------------------------------
# Generate date sequence
# --------------------------------------------

date_df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 day
            )
        ) AS calendar_date
    """)
)

# --------------------------------------------
# Build dim_date
# --------------------------------------------

dim_date = (
    date_df
    .withColumn("date_key", date_format(col("calendar_date"), "yyyyMMdd").cast("int"))
    .withColumn("day", dayofmonth(col("calendar_date")))
    .withColumn("month", month(col("calendar_date")))
    .withColumn("month_name", date_format(col("calendar_date"), "MMMM"))
    .withColumn("quarter", quarter(col("calendar_date")))
    .withColumn("year", year(col("calendar_date")))
    .withColumn("day_of_week", dayofweek(col("calendar_date")))
    .withColumn("is_weekend", col("day_of_week").isin([1, 7]))
    .withColumn("is_business_day", ~col("is_weekend"))
)

# --------------------------------------------
# Write Gold dim_date
# --------------------------------------------

(
    dim_date
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lh_Gold_Telecom.dbo.dim_date")
)

print("✓ dim_date created successfully")


StatementMeta(, ce1b4d7c-12fd-4a8e-89e8-648e62ac6e10, 3, Finished, Available, Finished)

Generating dim_date from 2024-01-01 to 2024-04-28
✓ dim_date created successfully
